# Notebook for demonstrating and testing MAS capabilities of data retrievemnt

The use of individual workflows of this system is also available to users and developers. 

This journal will demonstrate how to use workflows to retrive data from data lake using text requests.

In [ ]:
import sys
sys.path.append('./../src')

## 0. Demontrating task example

In [ ]:
from dotenv import load_dotenv
import json
import os
import shutil

from datalake_driver.drivers.local_storage_driver import LocalDriver
from data_model.datalake import Datalake

load_dotenv('./../src/.env')

In [ ]:
with open('raw_testing_material/imgs_metadata_rich_descr.json', 'r') as i_stream:
    some_dict = json.load(i_stream)

if os.path.exists('./testing_datalake'):
    shutil.rmtree('./testing_datalake')

local_config = {
    'root_folder': './'
}

driver = LocalDriver(**local_config)

name = 'testing_datalake'
local_datalake = Datalake(name, name + '/', driver)

In [ ]:
with open('raw_testing_material/imgs_metadata_rich_descr.json', 'r') as i_stream:
    some_dict = json.load(i_stream)

local_datalake.create_dataset('fluorescent_microscopy', '3d fluorescent images of neurons, captured by confocal microscope', some_dict)

In [ ]:
from copy import deepcopy

with open('raw_testing_material/new_data_values.json', 'r') as i_stream:
    new_data_meta_vals = json.load(i_stream)
new_data_meta_vals

small_data_meta = deepcopy(new_data_meta_vals)
small_data_meta['number_of_pixels_x'] = 128
small_data_meta['number_of_pixels_y'] = 128

local_datalake.add_local_file('fluorescent_microscopy', small_data_meta, 'mockup_img1.tiff', 'raw_testing_material/imgs/')
local_datalake.add_local_file('fluorescent_microscopy', new_data_meta_vals, 'mockup_img2.tiff', 'raw_testing_material/imgs/')
local_datalake.add_local_file('fluorescent_microscopy', small_data_meta, 'mockup_img3.tiff', 'raw_testing_material/imgs/')

In [ ]:
TEST_TASK_1 = "Hi! I need some data captured by fluorescent microscope, where pixel sizes along OX and OY axies are more then 1024 an less then 3096. Can you get it from me?"

This example shows how to use an SQL query to get images with a layer resolution from 1025 to 4095 pixels.

In [ ]:
sql_request = 'SELECT * FROM df WHERE number_of_pixels_x > 1024 AND number_of_pixels_x < 4096 AND number_of_pixels_y > 1024 AND number_of_pixels_y < 4096'
df = local_datalake.get_data_table_sql('fluorescent_microscopy', sql_request)
df.head()

## 1. Demonstrating data retrieving

In [ ]:
import os
import yaml

prompts_names = ['getting_data_sp.yaml']

with open(os.path.join('../src/prompts_templates/getting_data', prompts_names[0])) as stream:
    get_data_sp = yaml.safe_load(stream)['system_prompt']

get_data_sp

In [ ]:
from mas_exec.scidatamas.data_retrieval_workflow import DataRetrievingFlow

add_flow = DataRetrievingFlow(get_data_sp, local_datalake)

In [ ]:
from mas_exec.scidatamas.data_retrieval_workflow import DataRetrievingFlow, RetrievingDatalakeDataState

def get_data(user_task: str, datalake:Datalake, system_prompt: str, model: str, provider: str):
    add_data_workflow = DataRetrievingFlow(system_prompt=system_prompt, datalake=datalake, model=model, provider=provider)

    working_state = RetrievingDatalakeDataState()
    working_state['users_task'] = user_task
    wf = add_data_workflow.get_workflow()
    wf = wf.compile()
    working_state = wf.invoke(working_state)
    return working_state

In [ ]:
final_state = get_data(TEST_TASK_1, local_datalake, get_data_sp, model='mistral-large-latest', provider='mistralai')

Here is a result of workflow usage.

In [ ]:
final_state